### Playground: Regression Evaluation Metrics (MAE, MSE, RMSE, R², Adjusted R²)

Evaluating regression models requires quantifying how far predicted values ($\hat{y}$) deviate from actual target values ($y$). Different metrics emphasize different aspects of model error—such as sensitivity to extreme outliers or feature count penalties.

### Key Learning Objectives:
1. Understand and implement core regression error metrics: **MAE**, **MSE**, and **RMSE**.
2. Analyze the mathematical properties of **$R^2$** (Coefficient of Determination) and **Adjusted $R^2$**.
3. Learn how $R^2$ can be artificially inflated by adding noise features, and why **Adjusted $R^2$** prevents this trap.
4. Compare metrics using pure Python/NumPy vs. `scikit-learn.metrics`.


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

# Seed for reproducibility
np.random.seed(2026)

In [2]:
n_samples = 100
# Synthetic Real Estate Price Dataset
sqft = np.random.uniform(800, 3500, size=n_samples)
bedrooms = np.random.choice([1, 2, 3, 4], size=n_samples)
actual_price = 150000 + (180 * sqft) + (25000 * bedrooms) + np.random.normal(0, 35000, size=n_samples)

df_house = pd.DataFrame({
    "sqft": sqft.round(),
    "bedrooms": bedrooms,
    "actual_price": actual_price.round(2)
})
X = df_house[["sqft", "bedrooms"]]
y = df_house["actual_price"]
model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)
print("Actual Prices vs Predicted Prices")
print(pd.DataFrame({
    "Actual": y,
    "Predicted": y_pred,
    "Difference": y - y_pred
}).head())

Actual Prices vs Predicted Prices
      Actual      Predicted    Difference
0  437124.98  454677.980979 -17553.000979
1  557824.14  547611.593595  10212.546405
2  876693.57  841065.534174  35628.035826
3  390331.12  369125.238796  21205.881204
4  633302.69  625428.078680   7874.611320


---
#### 1. Absolute & Squared Error Metrics (MAE, MSE, RMSE)

### Mathematical Formulas:
* **Mean Absolute Error (MAE)**:
  $$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$
  *Interpretable in original target units (₹/$) and robust to outliers.*

* **Mean Squared Error (MSE)**:
  $$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$
  *Heavily penalizes large residual errors due to squaring.*

* **Root Mean Squared Error (RMSE)**:
  $$\text{RMSE} = \sqrt{\text{MSE}} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$
  *Brings the squared penalty scale back into original target units.*

In [3]:
# 1. Manual Calculations via NumPy
mae_manual = np.mean(np.abs(y - y_pred))
mse_manual = np.mean((y - y_pred) ** 2)
rmse_manual = np.sqrt(mse_manual)

# 2. Scikit-Learn Metrics
mae_sklearn = mean_absolute_error(y, y_pred)
mse_sklearn = mean_squared_error(y, y_pred)
rmse_sklearn = root_mean_squared_error(y, y_pred)  # Scikit-Learn 1.4+ API

print("--- Evaluation Metric Comparison ---")
print(f"MAE  -> Manual: ₹{mae_manual:10.2f} | Scikit-Learn: ₹{mae_sklearn:10.2f}")
print(f"MSE  -> Manual: ₹{mse_manual:10.2f} | Scikit-Learn: ₹{mse_sklearn:10.2f}")
print(f"RMSE -> Manual: ₹{rmse_manual:10.2f} | Scikit-Learn: ₹{rmse_sklearn:10.2f}")

--- Evaluation Metric Comparison ---
MAE  -> Manual: ₹  29283.79 | Scikit-Learn: ₹  29283.79
MSE  -> Manual: ₹1295331987.74 | Scikit-Learn: ₹1295331987.74
RMSE -> Manual: ₹  35990.72 | Scikit-Learn: ₹  35990.72


---
## 2. Goodness-of-Fit: $R^2$ and Adjusted $R^2$

### Mathematical Formulas:
* **$R^2$ (Coefficient of Determination)**:
  $$R^2 = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}} = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$
  *Measures the proportion of total variance in the target explained by feature predictors ($0 \le R^2 \le 1$).*

* **Adjusted $R^2$**:
  $$R^2_{\text{adj}} = 1 - \left[ \frac{(1 - R^2)(n - 1)}{n - p - 1} \right]$$
  *Penalizes $R^2$ for adding non-informative predictor features ($n = \text{sample count}, p = \text{feature count}$).*

In [ ]:
# ============================================================
# Assignment: Evaluation Metrics for Real Estate Price Prediction
# ============================================================
# [AUDIT OF WHAT WAS MISSING / WRONG IN PULLED CODE]:
# 1. Cell stopped abruptly with only a header comment.
# 2. Key Learning Objective 2 ($R^2$ vs Adjusted $R^2$) was completely unimplemented.
# 3. Key Learning Objective 3 (The Noise Feature Trap & $R^2$ inflation) was missing.
# Below is the verified, robust, and complete implementation.
# ============================================================

# ------------------------------------------------------------
# Step 1: Calculate R² (Coefficient of Determination)
# Formula: R² = 1 - (SS_res / SS_tot)
# ------------------------------------------------------------
y_mean = np.mean(y)
ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - y_mean) ** 2)
r2_manual = 1 - (ss_res / ss_tot)
r2_sklearn = r2_score(y, y_pred)

print("--- Step 1: R² Score Comparison ---")
print(f"R² (Manual NumPy)   : {r2_manual:.4f}")
print(f"R² (Scikit-Learn)   : {r2_sklearn:.4f}")
print("=" * 60)

# ------------------------------------------------------------
# Step 2: Calculate Adjusted R²
# Formula: R²_adj = 1 - [((1 - R²)(n - 1)) / (n - p - 1)]
# where n = sample size, p = number of predictor features
# ------------------------------------------------------------
n = len(y)
p = X.shape[1]  # 2 features: sqft and bedrooms

def calculate_adj_r2(r2, n, p):
    '''
    Computes Adjusted R² to penalize non-informative features.
    Prevents denominator division errors if n - p - 1 <= 0.
    '''
    if n - p - 1 <= 0:
        return np.nan
    return 1 - ((1 - r2) * (n - 1) / (n - p - 1))

adj_r2_baseline = calculate_adj_r2(r2_sklearn, n, p)

print("--- Step 2: Adjusted R² (Baseline Model) ---")
print(f"Sample Count (n)     : {n}")
print(f"Feature Count (p)    : {p} (sqft, bedrooms)")
print(f"Adjusted R²          : {adj_r2_baseline:.4f}")
print("=" * 60)

# ------------------------------------------------------------
# Step 3: Proving the 'Noise Feature Trap' (Objective 3)
# WHY R² ALONE IS DECEPTIVE:
# Mathematically, adding any feature (even pure random noise) will
# NEVER decrease standard R². It either stays equal or goes up.
# Adjusted R² applies a penalty (n - 1)/(n - p - 1), so useless
# features cause Adjusted R² to DROP.
# ------------------------------------------------------------
np.random.seed(42)
n_noise_features = 5
noise_cols = [f"noise_feature_{i+1}" for i in range(n_noise_features)]

# Add 5 pure Gaussian noise columns with ZERO relationship to price
df_noisy = df_house.copy()
for col in noise_cols:
    df_noisy[col] = np.random.normal(0, 1, size=n)

X_noisy = df_noisy[["sqft", "bedrooms"] + noise_cols]
p_noisy = X_noisy.shape[1]  # 2 real + 5 noise = 7 features

model_noisy = LinearRegression()
model_noisy.fit(X_noisy, y)
y_pred_noisy = model_noisy.predict(X_noisy)

r2_noisy = r2_score(y, y_pred_noisy)
adj_r2_noisy = calculate_adj_r2(r2_noisy, n, p_noisy)

print("--- Step 3: Demonstrating the R² Inflation Trap ---")
print(f"Baseline Model (p=2) -> R²: {r2_sklearn:.4f} | Adjusted R²: {adj_r2_baseline:.4f}")
print(f"Noisy Model    (p=7) -> R²: {r2_noisy:.4f} | Adjusted R²: {adj_r2_noisy:.4f}")

print("\n[OBSERVATION & INTERVIEW INSIGHT]:")
print(f"1. Standard R² increased from {r2_sklearn:.4f} to {r2_noisy:.4f} (False improvement!).")
print(f"2. Adjusted R² decreased from {adj_r2_baseline:.4f} to {adj_r2_noisy:.4f} (Correctly penalizes noise!).")
print("=" * 60)

# ------------------------------------------------------------
# Step 4: Summary Comparison Table & Visualization
# ------------------------------------------------------------
summary_metrics = pd.DataFrame({
    "Model": ["Baseline (sqft, bedrooms)", "Noisy (+5 Random Features)"],
    "Features (p)": [p, p_noisy],
    "MAE (₹)": [mean_absolute_error(y, y_pred), mean_absolute_error(y, y_pred_noisy)],
    "RMSE (₹)": [root_mean_squared_error(y, y_pred), root_mean_squared_error(y, y_pred_noisy)],
    "R² Score": [r2_sklearn, r2_noisy],
    "Adjusted R²": [adj_r2_baseline, adj_r2_noisy]
})
print("--- Step 4: Comprehensive Model Comparison ---")
print(summary_metrics.to_string(index=False))
print("=" * 60)

# Visualizing Baseline vs Noisy Model
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(y, y_pred, color='steelblue', alpha=0.7)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
plt.title(f"Baseline (R²={r2_sklearn:.3f}, Adj R²={adj_r2_baseline:.3f})")
plt.xlabel("Actual Price (₹)")
plt.ylabel("Predicted Price (₹)")

plt.subplot(1, 2, 2)
plt.scatter(y, y_pred_noisy, color='darkorange', alpha=0.7)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
plt.title(f"Noisy (+5 Noise) (R²={r2_noisy:.3f}, Adj R²={adj_r2_noisy:.3f})")
plt.xlabel("Actual Price (₹)")
plt.tight_layout()
plt.show()
